In [1]:
from pathlib import Path
CARPETA_DIA_19 = Path("dia_19_supabase_fastapi")
CARPETA_DIA_19.mkdir(exist_ok=True)

print("Carpeta creada:")
print(CARPETA_DIA_19.resolve())

Carpeta creada:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\dia_19_supabase_fastapi


In [2]:
requirements = """
fastapi
uvicorn[standard]
supabase
python-dotenv
requests
pandas
matplotlib
"""

(CARPETA_DIA_19 / "requirements.txt").write_text(
    requirements.strip(),
    encoding="utf-8"
)

print("requirements.txt creado.")

requirements.txt creado.


In [ ]:
env_example = """
SUPABASE_URL=
SUPABASE_KEY=
"""

(CARPETA_DIA_19 / ".env.example").write_text(
    env_example.strip(),
    encoding="utf-8"
)

print(".env.example creado.")

.env.example creado.


In [4]:
gitignore = """
.env
__pycache__/
*.pyc
.venv/
venv/
.ipynb_checkpoints/
"""

(CARPETA_DIA_19 / ".gitignore").write_text(
    gitignore.strip(),
    encoding="utf-8"
)

print(".gitignore creado.")

.gitignore creado.


In [5]:
codigo_supabase_client = r'''
import os
from dotenv import load_dotenv
from supabase import create_client, Client


load_dotenv()


SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")


if not SUPABASE_URL:
    raise ValueError("Falta SUPABASE_URL en el archivo .env")

if not SUPABASE_KEY:
    raise ValueError("Falta SUPABASE_KEY en el archivo .env")


supabase: Client = create_client(
    SUPABASE_URL,
    SUPABASE_KEY
)
'''

(CARPETA_DIA_19 / "supabase_client.py").write_text(
    codigo_supabase_client,
    encoding="utf-8"
)

print("supabase_client.py creado.")

supabase_client.py creado.


In [6]:
codigo_test = r'''
from supabase_client import supabase


def main():
    respuesta = (
        supabase
        .table("productos")
        .select("*")
        .limit(5)
        .execute()
    )

    print("Datos recibidos:")
    print(respuesta.data)


if __name__ == "__main__":
    main()
'''

(CARPETA_DIA_19 / "test_supabase.py").write_text(
    codigo_test,
    encoding="utf-8"
)

print("test_supabase.py creado.")

test_supabase.py creado.


In [7]:
codigo_services_supabase = r'''
from supabase_client import supabase


def respuesta_ok(mensaje, **datos):
    respuesta = {
        "ok": True,
        "mensaje": mensaje
    }

    respuesta.update(datos)

    return respuesta


def respuesta_error(tipo_error, mensaje):
    return {
        "ok": False,
        "tipo_error": tipo_error,
        "mensaje": mensaje
    }


def limpiar_texto(valor):
    if valor is None:
        return ""

    return str(valor).strip()


def normalizar_sku(sku):
    return limpiar_texto(sku).upper()


def normalizar_correo(correo):
    return limpiar_texto(correo).lower()


def validar_texto_obligatorio(valor, nombre_campo):
    valor = limpiar_texto(valor)

    if valor == "":
        raise ValueError(f"El campo {nombre_campo} no puede estar vacío.")

    return valor


def validar_precio(precio):
    try:
        precio = float(precio)
    except (TypeError, ValueError):
        raise ValueError("El precio debe ser numérico.")

    if precio <= 0:
        raise ValueError("El precio debe ser mayor que cero.")

    return precio


def validar_stock(stock):
    try:
        stock = int(stock)
    except (TypeError, ValueError):
        raise ValueError("El stock debe ser entero.")

    if stock < 0:
        raise ValueError("El stock no puede ser negativo.")

    return stock


# ---------------------------------------------------------
# PRODUCTOS
# ---------------------------------------------------------

def listar_productos(solo_activos=True):
    consulta = (
        supabase
        .table("productos")
        .select("*")
        .order("nombre")
    )

    if solo_activos:
        consulta = consulta.eq("activo", True)

    respuesta = consulta.execute()

    return respuesta.data


def buscar_producto_por_sku(sku):
    sku = normalizar_sku(sku)

    respuesta = (
        supabase
        .table("productos")
        .select("*")
        .eq("sku", sku)
        .limit(1)
        .execute()
    )

    if not respuesta.data:
        return None

    return respuesta.data[0]


def registrar_producto(sku, nombre, categoria, precio, stock):
    try:
        sku = normalizar_sku(sku)
        nombre = validar_texto_obligatorio(nombre, "nombre")
        categoria = validar_texto_obligatorio(categoria, "categoría")
        precio = validar_precio(precio)
        stock = validar_stock(stock)

        existente = buscar_producto_por_sku(sku)

        if existente is not None:
            return respuesta_error(
                "conflicto",
                "Ya existe un producto con ese SKU."
            )

        nuevo_producto = {
            "sku": sku,
            "nombre": nombre,
            "categoria": categoria,
            "precio": precio,
            "stock": stock,
            "activo": True
        }

        respuesta = (
            supabase
            .table("productos")
            .insert(nuevo_producto)
            .execute()
        )

        producto = respuesta.data[0] if respuesta.data else nuevo_producto

        return respuesta_ok(
            "Producto registrado correctamente.",
            producto=producto
        )

    except ValueError as error:
        return respuesta_error("validacion", str(error))

    except Exception as error:
        return respuesta_error("base_datos", str(error))


def actualizar_producto(sku, datos_actualizar):
    try:
        sku = normalizar_sku(sku)

        producto = buscar_producto_por_sku(sku)

        if producto is None:
            return respuesta_error(
                "no_encontrado",
                "Producto no encontrado."
            )

        datos = {}

        if "nombre" in datos_actualizar and datos_actualizar["nombre"] is not None:
            datos["nombre"] = validar_texto_obligatorio(
                datos_actualizar["nombre"],
                "nombre"
            )

        if "categoria" in datos_actualizar and datos_actualizar["categoria"] is not None:
            datos["categoria"] = validar_texto_obligatorio(
                datos_actualizar["categoria"],
                "categoría"
            )

        if "precio" in datos_actualizar and datos_actualizar["precio"] is not None:
            datos["precio"] = validar_precio(datos_actualizar["precio"])

        if "stock" in datos_actualizar and datos_actualizar["stock"] is not None:
            datos["stock"] = validar_stock(datos_actualizar["stock"])

        if "activo" in datos_actualizar and datos_actualizar["activo"] is not None:
            datos["activo"] = bool(datos_actualizar["activo"])

        if not datos:
            return respuesta_error(
                "validacion",
                "No se enviaron datos para actualizar."
            )

        respuesta = (
            supabase
            .table("productos")
            .update(datos)
            .eq("sku", sku)
            .execute()
        )

        producto_actualizado = respuesta.data[0] if respuesta.data else buscar_producto_por_sku(sku)

        return respuesta_ok(
            "Producto actualizado correctamente.",
            producto=producto_actualizado
        )

    except ValueError as error:
        return respuesta_error("validacion", str(error))

    except Exception as error:
        return respuesta_error("base_datos", str(error))


def desactivar_producto(sku):
    sku = normalizar_sku(sku)

    producto = buscar_producto_por_sku(sku)

    if producto is None:
        return respuesta_error(
            "no_encontrado",
            "Producto no encontrado."
        )

    respuesta = (
        supabase
        .table("productos")
        .update({"activo": False})
        .eq("sku", sku)
        .execute()
    )

    producto_actualizado = respuesta.data[0] if respuesta.data else buscar_producto_por_sku(sku)

    return respuesta_ok(
        "Producto desactivado correctamente.",
        producto=producto_actualizado
    )


# ---------------------------------------------------------
# CLIENTES
# ---------------------------------------------------------

def listar_clientes(solo_activos=True):
    consulta = (
        supabase
        .table("clientes")
        .select("*")
        .order("nombre")
    )

    if solo_activos:
        consulta = consulta.eq("activo", True)

    respuesta = consulta.execute()

    return respuesta.data


def buscar_cliente_por_id(id_cliente):
    respuesta = (
        supabase
        .table("clientes")
        .select("*")
        .eq("id_cliente", id_cliente)
        .limit(1)
        .execute()
    )

    if not respuesta.data:
        return None

    return respuesta.data[0]


def registrar_cliente(nombre, correo, telefono=None, ciudad="No especificada"):
    try:
        nombre = validar_texto_obligatorio(nombre, "nombre")
        correo = normalizar_correo(correo)

        if correo == "":
            raise ValueError("El correo no puede estar vacío.")

        cliente = {
            "nombre": nombre,
            "correo": correo,
            "telefono": telefono,
            "ciudad": ciudad or "No especificada",
            "activo": True
        }

        respuesta = (
            supabase
            .table("clientes")
            .insert(cliente)
            .execute()
        )

        cliente_insertado = respuesta.data[0] if respuesta.data else cliente

        return respuesta_ok(
            "Cliente registrado correctamente.",
            cliente=cliente_insertado
        )

    except ValueError as error:
        return respuesta_error("validacion", str(error))

    except Exception as error:
        mensaje = str(error)

        if "duplicate" in mensaje.lower() or "unique" in mensaje.lower():
            return respuesta_error(
                "conflicto",
                "Ya existe un cliente con ese correo."
            )

        return respuesta_error("base_datos", mensaje)


# ---------------------------------------------------------
# RESUMEN
# ---------------------------------------------------------

def resumen_general():
    productos = listar_productos(solo_activos=True)
    clientes = listar_clientes(solo_activos=True)

    valor_inventario = 0

    for producto in productos:
        precio = float(producto.get("precio", 0))
        stock = int(producto.get("stock", 0))
        valor_inventario += precio * stock

    return {
        "productos_activos": len(productos),
        "clientes_activos": len(clientes),
        "valor_inventario": valor_inventario
    }
'''

(CARPETA_DIA_19 / "services_supabase.py").write_text(
    codigo_services_supabase,
    encoding="utf-8"
)

print("services_supabase.py creado.")

services_supabase.py creado.


In [8]:
codigo_main_supabase = r'''
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field
from typing import Optional

import services_supabase as services


app = FastAPI(
    title="API Sistema de Ventas con Supabase",
    description="API FastAPI conectada a Supabase/PostgreSQL.",
    version="3.0.0"
)


class ProductoCrear(BaseModel):
    sku: str = Field(min_length=1)
    nombre: str = Field(min_length=1)
    categoria: str = Field(min_length=1)
    precio: float = Field(gt=0)
    stock: int = Field(ge=0)


class ProductoActualizar(BaseModel):
    nombre: Optional[str] = None
    categoria: Optional[str] = None
    precio: Optional[float] = Field(default=None, gt=0)
    stock: Optional[int] = Field(default=None, ge=0)
    activo: Optional[bool] = None


class ClienteCrear(BaseModel):
    nombre: str = Field(min_length=1)
    correo: str = Field(min_length=1)
    telefono: Optional[str] = None
    ciudad: Optional[str] = "No especificada"


def modelo_a_dict(modelo, exclude_unset=False):
    if hasattr(modelo, "model_dump"):
        return modelo.model_dump(exclude_unset=exclude_unset)

    return modelo.dict(exclude_unset=exclude_unset)


def manejar_resultado_servicio(resultado):
    if resultado.get("ok"):
        return resultado

    tipo_error = resultado.get("tipo_error")
    mensaje = resultado.get("mensaje", "Error en la operación.")

    if tipo_error == "no_encontrado":
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=mensaje
        )

    if tipo_error == "conflicto":
        raise HTTPException(
            status_code=status.HTTP_409_CONFLICT,
            detail=mensaje
        )

    if tipo_error == "validacion":
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail=mensaje
        )

    raise HTTPException(
        status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
        detail=mensaje
    )


@app.get("/")
def inicio():
    return {
        "mensaje": "API conectada a Supabase funcionando.",
        "base_datos": "Supabase PostgreSQL",
        "documentacion": "/docs"
    }


@app.get("/health")
def health():
    resumen = services.resumen_general()

    return {
        "status": "ok",
        "base_datos": "Supabase PostgreSQL",
        "resumen": resumen
    }


@app.get("/productos")
def listar_productos(solo_activos: bool = True):
    return services.listar_productos(solo_activos=solo_activos)


@app.get("/productos/{sku}")
def obtener_producto(sku: str):
    producto = services.buscar_producto_por_sku(sku)

    if producto is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Producto no encontrado."
        )

    return producto


@app.post("/productos", status_code=status.HTTP_201_CREATED)
def crear_producto(datos: ProductoCrear):
    datos_producto = modelo_a_dict(datos)

    resultado = services.registrar_producto(
        sku=datos_producto["sku"],
        nombre=datos_producto["nombre"],
        categoria=datos_producto["categoria"],
        precio=datos_producto["precio"],
        stock=datos_producto["stock"]
    )

    return manejar_resultado_servicio(resultado)


@app.put("/productos/{sku}")
def actualizar_producto(sku: str, datos: ProductoActualizar):
    datos_actualizar = modelo_a_dict(datos, exclude_unset=True)

    resultado = services.actualizar_producto(
        sku=sku,
        datos_actualizar=datos_actualizar
    )

    return manejar_resultado_servicio(resultado)


@app.delete("/productos/{sku}")
def desactivar_producto(sku: str):
    resultado = services.desactivar_producto(sku)

    return manejar_resultado_servicio(resultado)


@app.get("/clientes")
def listar_clientes(solo_activos: bool = True):
    return services.listar_clientes(solo_activos=solo_activos)


@app.get("/clientes/{id_cliente}")
def obtener_cliente(id_cliente: int):
    cliente = services.buscar_cliente_por_id(id_cliente)

    if cliente is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Cliente no encontrado."
        )

    return cliente


@app.post("/clientes", status_code=status.HTTP_201_CREATED)
def crear_cliente(datos: ClienteCrear):
    datos_cliente = modelo_a_dict(datos)

    resultado = services.registrar_cliente(
        nombre=datos_cliente["nombre"],
        correo=datos_cliente["correo"],
        telefono=datos_cliente.get("telefono"),
        ciudad=datos_cliente.get("ciudad")
    )

    return manejar_resultado_servicio(resultado)


@app.get("/resumen")
def resumen():
    return services.resumen_general()
'''

(CARPETA_DIA_19 / "main_supabase.py").write_text(
    codigo_main_supabase,
    encoding="utf-8"
)

print("main_supabase.py creado.")

main_supabase.py creado.
